# Bootstrap


In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "pyproject.toml").is_file() and (p / "research" / "lib").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot find Tyrex_PM repo root. Start Jupyter from repo root or research/notebooks/."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 1. Objective
Audit coverage, quality, and label validity. Produce `clean_markets.csv` gate for notebooks 02-06.

## 2. Strategy relevance
Informs which markets are safe for offline research and which labels (direction vs price-to-beat) are proxy-valid.

## 3. Data used
- `markets.parquet`: one row per BTC 5m market window
- `ws_quality.parquet`, `lifecycle_events`: gap/staleness diagnostics
- `price_to_beat.parquet`: settlement reference labels
Caveat: gap_rate is diagnostic only — not a live strategy feature.

## 4. Key formulas
- `gap_rate = ws_seq_gap_count / event_count` (diagnostic)
- `include_for_analysis` requires PTB present, zero corrupt/dropped, coverage recorded


### Load partition and build clean-market gate


In [2]:
from research.lib.loaders import load_day
from research.lib.markets import build_clean_markets, notebook_01_decisions, ws_seq_gap_audit, write_clean_markets
from research.lib.exploratory import write_exploratory_trace
from research.m2b4.exploratory import run_notebook_01_exploratory
from research.lib.buckets import format_decision_output
from research.lib.plots import try_import_matplotlib

PARTITION = REPO_ROOT / "var/parquet/date=2026-07-05"
OUT = REPO_ROOT / "research/output/m2b4"
PLOT = OUT / "plots"
OUT.mkdir(parents=True, exist_ok=True)
PLOT.mkdir(parents=True, exist_ok=True)
day = load_day(PARTITION, load_books=False)
clean = build_clean_markets(day)
write_clean_markets(clean, OUT / "clean_markets.csv")
gap_audit = ws_seq_gap_audit(day.tables.get("lifecycle_events"))


### Visual exploration
Bar chart of included vs excluded markets. Do not conclude live readiness from counts alone.


In [ ]:
plt = try_import_matplotlib()
if plt:
    import matplotlib.pyplot as plt
    inc = int(clean["include_for_analysis"].sum())
    exc = len(clean) - inc
    plt.figure(figsize=(5,3))
    plt.bar(["included","excluded"], [inc, exc])
    plt.title("Clean markets (exploratory)")
    plt.savefig(PLOT / "01_included_excluded_bar.png")
    plt.close()
    clean["gap_rate"].hist(bins=20)
    plt.title("gap_rate distribution (diagnostic only)")
    plt.savefig(PLOT / "01_gap_rate_distribution.png")
    plt.close()
    clean.groupby("ptb_present")["market_id"].count().plot(kind="bar", title="PTB presence")
    plt.savefig(PLOT / "01_ptb_final_reference.png")
    plt.close()


plotting


## 6. EXPLORATORY TRACE
Exploratory only — not for live YAML.


In [5]:
exp = run_notebook_01_exploratory(day, clean, gap_audit)
write_exploratory_trace("01", exp, OUT)
print("gap_rate_usable_as_feature (exploratory):", exp.get("gap_rate_usable_as_feature"))


gap_rate_usable_as_feature (exploratory): False


## 7. STRICT DECISION OUTPUT


In [ ]:
decisions = notebook_01_decisions(day, clean, gap_audit)
print(format_decision_output(
    decisions=decisions, confidence="low", total_sample_size=len(clean),
    per_bucket_samples={{"included": int(clean["include_for_analysis"].sum())}},
    provisional=True, provisional_reason="27-market sample",
    follow_up="Notebooks 02-06 require clean_markets.csv",
))
clean.head()


## 8. Conclusions
- Learned: most markets pass PTB gate; one pre-fix exclusion expected.
- Unsafe: using gap_rate as a strategy feature.
- Unlocks confidence: multi-day recorder + M2B.1-B gate.
